In [51]:
!cp "/content/drive/MyDrive/Hate Speech LLMs/Data ICWSM/Eval Data/deepSeekR1Data.xlsx" .

In [52]:
import numpy as np
import pandas as pd
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score

In [53]:
df = pd.read_excel("deepSeekR1Data.xlsx")

In [54]:
ref_col = 'Hop_2_Target_Basis_of_Identity (Ref)'
pred_col = 'Hop_2_Target_Basis_of_Identity'

In [55]:
df = df[df["Hop_1_Triage (Ref)"] != "بی‌ضرر"]
df = df[df["Hop_1_Triage"] != "بی‌ضرر"]
df[pred_col] = df[pred_col].fillna('[]')
print(len(df))

872


In [56]:
refIdentities = []
for refIdentity in df[ref_col]:
  refIdentities.extend(ast.literal_eval(refIdentity))
set(refIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سوابق تحصیلی/دانشگاهی',
 'طبقه اجتماعی',
 'عضویت گروهی',
 'قومیت/ملیت',
 'ماهیت انسانی',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [57]:
llmIdentities = []
for llmIdentity in df[pred_col]:
  llmIdentities.extend(ast.literal_eval(llmIdentity))
set(llmIdentities)

{'other',
 'جنسیت',
 'حرفه/شغل',
 'دین/باور',
 'سن',
 'سوابق تحصیلی/دانشگاهی',
 'طبقه اجتماعی',
 'عضویت گروهی',
 'قومیت/ملیت',
 'ماهیت انسانی',
 'ملیت',
 'ملیت/قومیت',
 'نقش اجتماعی',
 'نقش اجتماعی/خانوادگی',
 'هویت فردی',
 'وابستگی سازمانی/نهادی',
 'وضعیت مهاجرت',
 'ویژگی ذهنی/رفتاری',
 'گرایش سیاسی'}

In [58]:
len(set(refIdentities))

15

In [59]:
len(set(llmIdentities))

19

In [60]:
df[ref_col] = df[ref_col].apply(ast.literal_eval)
df[pred_col] = df[pred_col].apply(ast.literal_eval)

In [61]:
df[ref_col] = df[ref_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df[pred_col] = df[pred_col].apply(
    lambda x: [label for label in x if label != 'other']
)

df_eval = df[
    df[ref_col].apply(len).gt(0) &
    df[pred_col].apply(len).gt(0)
].copy()

In [62]:
mlb = MultiLabelBinarizer()

mlb.fit(list(df_eval[ref_col]) + list(df_eval[pred_col]))

y_true = mlb.transform(df_eval[ref_col])
y_pred = mlb.transform(df_eval[pred_col])

In [63]:
f1_micro = f1_score(y_true, y_pred, average='micro')
f1_macro = f1_score(y_true, y_pred, average='macro')

print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")

Micro F1: 0.6705
Macro F1: 0.3956
